In [ ]:
!pip install gensim
!python -m spacy download pt_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 71.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
#LAB 03: Expansão de Classe (Data Drift & Novas Intenções)

# BLOCO 1: Preparação do Ambiente

import re
import numpy as np
import pandas as pd
import spacy
import gensim.downloader as api
import gradio as gr

from sklearn.linear_model import LogisticRegression


# CARREGAMENTO DOS MODELOS

print("Carregando modelo morfológico Spacy...")

nlp = spacy.load(
    "pt_core_news_sm"
)

print("Carregando GloVe...")

word_vectors = api.load(
    "glove-wiki-gigaword-50"
)


# BLOCO 1: DATASET

dados_imobiliaria = [

    # --------------------------------------------------------
    # INTENÇÃO: comprar_imovel
    # --------------------------------------------------------

    (
        "Quero comprar um apartamento de 3 quartos com varanda",
        "comprar_imovel"
    ),

    (
        "Gostaria de ver casas à venda no centro da cidade",
        "comprar_imovel"
    ),

    (
        "Qual o preço médio para compra de cobertura com piscina?",
        "comprar_imovel"
    ),

    (
        "Procuro imóvel residencial para comprar com financiamento",
        "comprar_imovel"
    ),

    (
        "Vocês têm sobrados à venda na zona sul?",
        "comprar_imovel"
    ),


    # --------------------------------------------------------
    # INTENÇÃO: alugar_imovel
    # --------------------------------------------------------

    (
        "Procurando kitnet para alugar perto da faculdade",
        "alugar_imovel"
    ),

    (
        "Qual o valor do aluguel deste apartamento de 2 dormitórios?",
        "alugar_imovel"
    ),

    (
        "Quero alugar um galpão comercial para minha empresa",
        "alugar_imovel"
    ),

    (
        "Quais imóveis estão disponíveis para locação imediata?",
        "alugar_imovel"
    ),

    (
        "Preciso de uma casa para alugar que aceite animais",
        "alugar_imovel"
    ),


    # --------------------------------------------------------
    # INTENÇÃO: suporte_manutencao
    # --------------------------------------------------------

    (
        "O chuveiro do apartamento alugado queimou, como pedir conserto?",
        "suporte_manutencao"
    ),

    (
        "Muro da casa está com infiltração e vazamento de água",
        "suporte_manutencao"
    ),

    (
        "Preciso do contato do encanador para reparo na cozinha",
        "suporte_manutencao"
    ),

    (
        "A porta da varanda quebrou, quem faz a manutenção?",
        "suporte_manutencao"
    ),

    (
        "Vazamento no teto do banheiro precisa de reparo urgente",
        "suporte_manutencao"
    ),


    # --------------------------------------------------------
    # INTENÇÃO: 2via_boleto_contrato
    # --------------------------------------------------------

    (
        "Como faço para baixar a segunda via do boleto do aluguel?",
        "2via_boleto_contrato"
    ),

    (
        "Não recebi o boleto deste mês para pagamento",
        "2via_boleto_contrato"
    ),

    (
        "Preciso do informe de rendimentos e cópia do contrato",
        "2via_boleto_contrato"
    ),

    (
        "Onde pego o boleto atualizado com o valor do condomínio?",
        "2via_boleto_contrato"
    ),

    (
        "Quero solicitar a segunda via do recibo de pagamento",
        "2via_boleto_contrato"
    ),


    # --------------------------------------------------------
    # NOVA INTENÇÃO: cancelar_contrato
    # LAB03
    # --------------------------------------------------------

    (
        "Quero cancelar meu contrato de aluguel",
        "cancelar_contrato"
    ),

    (
        "Como faço para rescindir o contrato do imóvel?",
        "cancelar_contrato"
    ),

    (
        "Preciso solicitar o distrato do contrato",
        "cancelar_contrato"
    ),

    (
        "Quero encerrar meu contrato de locação",
        "cancelar_contrato"
    ),

    (
        "Gostaria de saber como cancelar o contrato",
        "cancelar_contrato"
    )
]


# DATAFRAME

df = pd.DataFrame(
    dados_imobiliaria,
    columns=[
        "mensagem",
        "intencao"
    ]
)

print(
    f"Dataset carregado com {len(df)} mensagens "
    f"divididas em {df['intencao'].nunique()} intenções."
)

print("\nQuantidade de exemplos por intenção:")
print(
    df["intencao"].value_counts()
)


# BLOCO 2: PRÉ-PROCESSAMENTO

def preprocessar_texto(texto: str) -> str:

    texto_limpo = texto.lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    doc = nlp(texto_limpo)

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_space
        and len(token.text) > 1
    ]

    return " ".join(tokens)


def extrair_sentence_embedding(
    texto_limpo: str,
    modelo_emb
) -> np.ndarray:

    palavras = texto_limpo.split()

    vetores = [
        modelo_emb[p]
        for p in palavras
        if p in modelo_emb
    ]

    if len(vetores) == 0:

        return np.zeros(
            modelo_emb.vector_size
        )

    return np.mean(
        vetores,
        axis=0
    )


# VETORIZAÇÃO DO DATASET

df["mensagem_limpa"] = df[
    "mensagem"
].apply(
    preprocessar_texto
)

X_densos = np.array([
    extrair_sentence_embedding(
        txt,
        word_vectors
    )
    for txt in df["mensagem_limpa"]
])

y = df["intencao"].values

print(
    "\nFormato da matriz X:",
    X_densos.shape
)

print(
    "Formato do vetor y:",
    y.shape
)


# BLOCO 3: TREINAMENTO

modelo_nlu = LogisticRegression(
    C=1.0,
    max_iter=500
)

modelo_nlu.fit(
    X_densos,
    y
)

print("\nModelo supervisionado treinado!")

print(
    "Classes aprendidas pelo modelo:"
)

print(
    modelo_nlu.classes_
)


# BASE DE RESPOSTAS

RESPOSTAS_PADRAO = {

    # --------------------------------------------------------
    # COMPRAR
    # --------------------------------------------------------

    "comprar_imovel": (
        "**Atendimento de Vendas:** Ficamos felizes "
        "com seu interesse! ")

}

Carregando modelo morfológico Spacy...
Carregando GloVe...
Dataset carregado com 25 mensagens divididas em 5 intenções.

Quantidade de exemplos por intenção:
intencao
comprar_imovel          5
alugar_imovel           5
suporte_manutencao      5
2via_boleto_contrato    5
cancelar_contrato       5
Name: count, dtype: int64

Formato da matriz X: (25, 50)
Formato do vetor y: (25,)

Modelo supervisionado treinado!
Classes aprendidas pelo modelo:
['2via_boleto_contrato' 'alugar_imovel' 'cancelar_contrato'
 'comprar_imovel' 'suporte_manutencao']
